# Advanced Feature Engineering (Day 13)

## Objective
Enrich the base feature set with advanced engineered features that capture **temporal patterns**, **feature interactions**, and **non-linear relationships** — then save as `h2_gold_model_scoring_engineered` for all downstream models.

## Feature Types Created

### 1. **Lag Features** (Historical Values)
* `price_lag_1h`, `price_lag_24h`, `price_lag_168h` (1 week)
* **Why**: Prices exhibit temporal autocorrelation

### 2. **Rolling Statistics** (Moving Averages)
* `price_rolling_mean_24h`, `price_rolling_std_24h`, min, max
* `load_rolling_mean_7d`, `renewable_rolling_mean_7d`
* **Why**: Smooth out noise, capture trends

### 3. **Interaction Features** (Combined Effects)
* `load_x_renewable`, `load_x_temperature`, `renewable_x_wind`
* **Why**: Features interact non-linearly

### 4. **Polynomial Features** (Non-Linear Relationships)
* `load_squared`, `temperature_squared`, `wind_speed_squared`
* **Why**: Extreme values have outsized impact

### 5. **Time-Based Features** (Enhanced)
* `is_weekend`, `is_peak_hour`, `season`

## Output
* Saves `h2_gold_model_scoring_engineered` — consumed by 09, 11, and other model notebooks
* Original 14 features + 19 engineered = **33 total features**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, PolynomialExpansion
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator, TrainValidationSplit
import mlflow
import numpy as np

print("✅ Imports loaded")

In [0]:
# Data configuration
CATALOG = "dbacademy"
SCHEMA  = "labuser13955035_1772876383"

# Source: FULL dataset (2020+2021) so downstream models have train AND test data
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.h2_gold_model_scoring_base"

# Target
TARGET_COL = "price_eur_mwh"
TIME_COL = "event_time_utc"
SPLIT_DATE = "2021-01-01"

# Output table
ENGINEERED_FEATURES_TABLE = f"{CATALOG}.{SCHEMA}.h2_gold_model_scoring_engineered"

# =============================================================================
# HYPERPARAMETER TUNING CONFIG
# =============================================================================
# On 0-worker cluster, keep grid small. Scale up on bigger clusters.
#
# SPEED GUIDE:
#   TrainValidationSplit = ~3x faster than CrossValidator (1 split vs k folds)
#   [20,50] trees + [3,5,8] depth = 6 combos × 1 split = 6 model fits
#   vs old: [50,100] × [5,10] × 3 folds = 12 fits with much heavier models
# =============================================================================
NUM_TREES_OPTIONS   = [20, 50]         # RF trees (20=fast baseline, 50=better)
MAX_DEPTH_OPTIONS   = [3, 5, 8]        # RF depth (3=fast, 8=rich but slower)
MIN_INFO_GAIN       = [0.0]            # RF min info gain
TRAIN_RATIO         = 0.8              # TrainValidationSplit: 80% train, 20% validation
METRIC              = "rmse"           # Evaluation metric: rmse, mae, r2

total_combos = len(NUM_TREES_OPTIONS) * len(MAX_DEPTH_OPTIONS) * len(MIN_INFO_GAIN)

print(f"Source:  {SOURCE_TABLE}")
print(f"Output:  {ENGINEERED_FEATURES_TABLE}")
print(f"\n--- Hyperparameter Search Space ---")
print(f"  num_trees:    {NUM_TREES_OPTIONS}")
print(f"  max_depth:    {MAX_DEPTH_OPTIONS}")
print(f"  min_info_gain:{MIN_INFO_GAIN}")
print(f"  Strategy:     TrainValidationSplit (ratio={TRAIN_RATIO})")
print(f"  Metric:       {METRIC}")
print(f"  Total fits:   {total_combos} combos × 1 split = {total_combos} models")
print(f"\n\u2705 Config ready — estimated ~2-5 min on 0-worker cluster")

In [0]:
print("Creating lag features...")
print("="*80)

# Load data
df = spark.table(SOURCE_TABLE).orderBy(TIME_COL)

# Define window for lag features (partition by zone, order by time)
window_spec = Window.partitionBy("zone").orderBy(TIME_COL)

# Lag features (1 hour, 24 hours, 1 week)
df_lag = df.withColumn(
    "price_lag_1h",
    F.lag(TARGET_COL, 1).over(window_spec)
).withColumn(
    "price_lag_24h",
    F.lag(TARGET_COL, 24).over(window_spec)
).withColumn(
    "price_lag_168h",  # 1 week
    F.lag(TARGET_COL, 168).over(window_spec)
).withColumn(
    "load_lag_1h",
    F.lag("avg_actual_total_load_mw", 1).over(window_spec)
).withColumn(
    "renewable_lag_1h",
    F.lag("renewable_generation_mw", 1).over(window_spec)
)

print(f"✅ Lag features created")
print(f"   - price_lag_1h, price_lag_24h, price_lag_168h")
print(f"   - load_lag_1h, renewable_lag_1h")

In [0]:
print("Creating rolling statistics...")
print("="*80)

# Rolling windows (24 hours, 7 days)
window_24h = Window.partitionBy("zone").orderBy(TIME_COL).rowsBetween(-23, 0)
window_7d = Window.partitionBy("zone").orderBy(TIME_COL).rowsBetween(-167, 0)

df_rolling = df_lag.withColumn(
    "price_rolling_mean_24h",
    F.avg(TARGET_COL).over(window_24h)
).withColumn(
    "price_rolling_std_24h",
    F.stddev(TARGET_COL).over(window_24h)
).withColumn(
    "price_rolling_min_24h",
    F.min(TARGET_COL).over(window_24h)
).withColumn(
    "price_rolling_max_24h",
    F.max(TARGET_COL).over(window_24h)
).withColumn(
    "load_rolling_mean_7d",
    F.avg("avg_actual_total_load_mw").over(window_7d)
).withColumn(
    "renewable_rolling_mean_7d",
    F.avg("renewable_generation_mw").over(window_7d)
)

print(f"✅ Rolling statistics created")
print(f"   - price_rolling_mean_24h, price_rolling_std_24h")
print(f"   - price_rolling_min_24h, price_rolling_max_24h")
print(f"   - load_rolling_mean_7d, renewable_rolling_mean_7d")

In [0]:
print("Creating interaction and polynomial features...")
print("="*80)

df_engineered = df_rolling.withColumn(
    # Interaction features
    "load_x_renewable",
    F.col("avg_actual_total_load_mw") * F.col("renewable_generation_mw")
).withColumn(
    "load_x_temperature",
    F.col("avg_actual_total_load_mw") * F.col("temperature_c")
).withColumn(
    "renewable_x_wind",
    F.col("renewable_generation_mw") * F.col("wind_speed_ms")
).withColumn(
    # Polynomial features
    "load_squared",
    F.pow(F.col("avg_actual_total_load_mw"), 2)
).withColumn(
    "temperature_squared",
    F.pow(F.col("temperature_c"), 2)
).withColumn(
    "wind_speed_squared",
    F.pow(F.col("wind_speed_ms"), 2)
).withColumn(
    # Enhanced time features
    "is_weekend",
    F.when(F.col("day_of_week").isin([6, 7]), 1).otherwise(0)
).withColumn(
    "is_peak_hour",
    F.when(F.col("hour_of_day").between(17, 21), 1).otherwise(0)
).withColumn(
    "season",
    F.when(F.col("month").isin([12, 1, 2]), "winter")
     .when(F.col("month").isin([3, 4, 5]), "spring")
     .when(F.col("month").isin([6, 7, 8]), "summer")
     .otherwise("fall")
)

print(f"✅ Interaction & polynomial features created")
print(f"   - Interactions: load_x_renewable, load_x_temperature, renewable_x_wind")
print(f"   - Polynomials: load_squared, temperature_squared, wind_speed_squared")
print(f"   - Enhanced time: is_weekend, is_peak_hour, season")

In [0]:
print("Handling missing values & saving...")
print("=" * 80)

# Lag/rolling features create NULLs at the beginning
initial_count = df_engineered.count()
df_clean = df_engineered.dropna()
final_count = df_clean.count()

print(f"Records before dropna: {initial_count:,}")
print(f"Records after dropna:  {final_count:,}")
print(f"Dropped: {initial_count - final_count:,} records ({(initial_count - final_count)/initial_count*100:.1f}%)")

# Save engineered features
df_clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(ENGINEERED_FEATURES_TABLE)

# Verify: must have BOTH 2020 and 2021 data
result_df = spark.table(ENGINEERED_FEATURES_TABLE)
train_rows = result_df.filter(F.col(TIME_COL) < SPLIT_DATE).count()
test_rows  = result_df.filter(F.col(TIME_COL) >= SPLIT_DATE).count()

print(f"\n\u2705 Saved to: {ENGINEERED_FEATURES_TABLE}")
print(f"   Total:  {result_df.count():,} rows, {len(result_df.columns)} columns")
print(f"   Train (pre-2021):  {train_rows:,} rows")
print(f"   Test  (2021+):     {test_rows:,} rows")

if test_rows == 0:
    print("\n\u274c WARNING: No test data!")
else:
    print(f"\n\u2705 Both train and test periods populated")

# Show sample
print(f"\nSample with new features:")
display(df_clean.select(
    TIME_COL, TARGET_COL,
    "price_lag_1h", "price_lag_24h",
    "price_rolling_mean_24h", "load_x_renewable",
    "is_weekend", "is_peak_hour"
).limit(10))

In [0]:
import time as _time

print("Preparing data for hyperparameter tuning...")
print("=" * 80)

# Load saved engineered features
df_eng = spark.table(ENGINEERED_FEATURES_TABLE)

# Train/test split
train = df_eng.filter(F.col(TIME_COL) < SPLIT_DATE)
test  = df_eng.filter(F.col(TIME_COL) >= SPLIT_DATE)

print(f"Train (pre-2021): {train.count():,} rows")
print(f"Test  (2021+):    {test.count():,} rows")

# Auto-detect numeric feature columns (exclude target, time, zone, string cols)
exclude = {TIME_COL, TARGET_COL, "zone", "high_price_period", "season"}
feature_cols = [
    f.name for f in df_eng.schema.fields
    if f.name not in exclude
    and f.dataType.simpleString() in ("double", "float", "int", "bigint", "integer")
]

print(f"\nFeature columns ({len(feature_cols)}):")
for c in feature_cols:
    print(f"  - {c}")

# Assembler
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

print(f"\n\u2705 Ready for tuning")

In [0]:
_t_tune = _time.time()
print("Running hyperparameter tuning (TrainValidationSplit)...")
print("=" * 80)
total_combos = len(NUM_TREES_OPTIONS) * len(MAX_DEPTH_OPTIONS) * len(MIN_INFO_GAIN)
print(f"Search space: {total_combos} combos × 1 split = {total_combos} model fits")
print(f"Strategy: TrainValidationSplit (80/20) — 3x faster than CrossValidator")
print()

# Model
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol=TARGET_COL,
    seed=42
)

# Parameterized grid
param_grid = ParamGridBuilder() \
    .addGrid(rf.numTrees, NUM_TREES_OPTIONS) \
    .addGrid(rf.maxDepth, MAX_DEPTH_OPTIONS) \
    .addGrid(rf.minInfoGain, MIN_INFO_GAIN) \
    .build()

# Evaluator
evaluator = RegressionEvaluator(
    labelCol=TARGET_COL,
    predictionCol="prediction",
    metricName=METRIC
)

# Pipeline
pipeline = Pipeline(stages=[assembler, rf])

# TrainValidationSplit (much faster than CrossValidator on single-node)
tvs = TrainValidationSplit(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    trainRatio=TRAIN_RATIO,
    parallelism=1,  # 0-worker: no benefit from parallelism
    seed=42
)

# Fit
with mlflow.start_run(run_name="rf_hyperparameter_tuning"):
    mlflow.log_params({
        "num_trees_options": str(NUM_TREES_OPTIONS),
        "max_depth_options": str(MAX_DEPTH_OPTIONS),
        "min_info_gain_options": str(MIN_INFO_GAIN),
        "strategy": "TrainValidationSplit",
        "train_ratio": TRAIN_RATIO,
        "num_features": len(feature_cols),
        "metric": METRIC
    })
    
    tvs_model = tvs.fit(train)
    
    # Best model metrics on held-out TEST set
    train_pred = tvs_model.transform(train)
    test_pred  = tvs_model.transform(test)
    
    train_score = evaluator.evaluate(train_pred)
    test_score  = evaluator.evaluate(test_pred)
    
    mlflow.log_metric(f"train_{METRIC}", train_score)
    mlflow.log_metric(f"test_{METRIC}", test_score)
    mlflow.spark.log_model(tvs_model.bestModel, "best_rf_model")

elapsed = _time.time() - _t_tune
print(f"\n\u2705 Tuning complete in {elapsed:.1f}s")
print(f"   Train {METRIC.upper()}: {train_score:.4f}")
print(f"   Test  {METRIC.upper()}: {test_score:.4f}")

In [0]:
print("Hyperparameter Tuning Results")
print("=" * 80)

# Extract results from TrainValidationSplit
val_metrics = tvs_model.validationMetrics

# Build results table
import pandas as pd

results = []
for i, params in enumerate(param_grid):
    row = {
        "num_trees": params[rf.numTrees],
        "max_depth": params[rf.maxDepth],
        "min_info_gain": params[rf.minInfoGain],
        f"val_{METRIC}": val_metrics[i]
    }
    results.append(row)

results_df = pd.DataFrame(results).sort_values(f"val_{METRIC}")
results_df["rank"] = range(1, len(results_df) + 1)
results_df = results_df[["rank", "num_trees", "max_depth", "min_info_gain", f"val_{METRIC}"]]

print("\n\u2705 All combinations ranked (lower RMSE = better):\n")
display(spark.createDataFrame(results_df))

# Best params
best_idx = val_metrics.index(min(val_metrics))
best_params = param_grid[best_idx]
print(f"\n\U0001f3c6 Best Parameters:")
print(f"   num_trees:     {best_params[rf.numTrees]}")
print(f"   max_depth:     {best_params[rf.maxDepth]}")
print(f"   min_info_gain: {best_params[rf.minInfoGain]}")
print(f"   Val {METRIC.upper()}:      {min(val_metrics):.4f}")
print(f"   Test {METRIC.upper()}:     {test_score:.4f}")

# Feature importance from best model
best_rf = tvs_model.bestModel.stages[-1]
if hasattr(best_rf, 'featureImportances'):
    importances = best_rf.featureImportances.toArray()
    imp_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": importances
    }).sort_values("importance", ascending=False)
    
    print(f"\n\u2705 Top 10 Feature Importances:")
    for _, row in imp_df.head(10).iterrows():
        bar = "\u2588" * int(row['importance'] * 50)
        print(f"   {row['feature']:35s} {row['importance']:.4f} {bar}")

In [0]:
from datetime import datetime

print("Saving best hyperparameters for downstream models...")
print("=" * 80)

# Best params from tuning
best_num_trees = int(best_params[rf.numTrees])
best_max_depth = int(best_params[rf.maxDepth])
best_min_info_gain = float(best_params[rf.minInfoGain])
best_val_rmse = float(min(val_metrics))
best_test_rmse = float(test_score)

# Save to Delta table
PARAMS_TABLE = f"{CATALOG}.{SCHEMA}.h2_tuning_best_params"

params_data = [{
    "model_type": "RandomForest",
    "task": "regression",
    "num_trees": best_num_trees,
    "max_depth": best_max_depth,
    "min_info_gain": best_min_info_gain,
    "val_rmse": best_val_rmse,
    "test_rmse": best_test_rmse,
    "num_features": len(feature_cols),
    "tuning_strategy": "TrainValidationSplit",
    "train_ratio": TRAIN_RATIO,
    "tuned_at": datetime.now().isoformat()
}]

spark.createDataFrame(params_data).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(PARAMS_TABLE)

print(f"\u2705 Best params saved to: {PARAMS_TABLE}")
print(f"   num_trees:     {best_num_trees}")
print(f"   max_depth:     {best_max_depth}")
print(f"   min_info_gain: {best_min_info_gain}")
print(f"   val_rmse:      {best_val_rmse:.4f}")
print(f"   test_rmse:     {best_test_rmse:.4f}")
print(f"\n\U0001f4e6 Downstream notebooks (09, 11) will read these params automatically")

In [0]:
_t_cv = _time.time()
NUM_FOLDS = 3  # 3 = fast (~90s), 5 = thorough (~150s)

print(f"Cross-validating best params with {NUM_FOLDS}-fold CV...")
print("=" * 80)
print(f"Best params from TVS: numTrees={best_num_trees}, maxDepth={best_max_depth}")
print(f"This validates the TVS result with proper k-fold — no grid search, just 1 combo × {NUM_FOLDS} folds")
print()

# Build model with ONLY the best params
rf_best = RandomForestRegressor(
    featuresCol="features",
    labelCol=TARGET_COL,
    numTrees=best_num_trees,
    maxDepth=best_max_depth,
    minInfoGain=best_min_info_gain,
    seed=42
)

pipeline_cv = Pipeline(stages=[assembler, rf_best])

# Single-point param grid (no search, just CV scoring)
param_grid_cv = ParamGridBuilder().build()  # empty = use model defaults above

# CrossValidator
cv = CrossValidator(
    estimator=pipeline_cv,
    estimatorParamMaps=param_grid_cv,
    evaluator=evaluator,
    numFolds=NUM_FOLDS,
    parallelism=1,
    seed=42
)

# Fit with k-fold
cv_model = cv.fit(train)

# Results
cv_avg_rmse = cv_model.avgMetrics[0]
cv_std_rmse = np.std([cv_model.avgMetrics[0]] * NUM_FOLDS)  # approximate

# Test set evaluation
cv_test_pred = cv_model.transform(test)
cv_test_rmse = evaluator.evaluate(cv_test_pred)

elapsed_cv = _time.time() - _t_cv

print(f"\n\u2705 {NUM_FOLDS}-Fold Cross-Validation Results:")
print(f"   CV Avg RMSE:    {cv_avg_rmse:.4f}")
print(f"   TVS Val RMSE:   {best_val_rmse:.4f}  (for comparison)")
print(f"   Test RMSE:      {cv_test_rmse:.4f}")
print(f"   \u23f1 Elapsed:      {elapsed_cv:.1f}s")

# Compare TVS vs CV
diff = abs(cv_avg_rmse - best_val_rmse)
if diff < 1.0:
    print(f"\n\u2705 TVS and CV scores are consistent (diff={diff:.4f}) — params are stable")
else:
    print(f"\n\u26a0\ufe0f  TVS and CV differ by {diff:.4f} — consider increasing NUM_FOLDS or TRAIN_RATIO")

# Update params table with CV score
from datetime import datetime
params_data_cv = [{
    "model_type": "RandomForest",
    "task": "regression",
    "num_trees": best_num_trees,
    "max_depth": best_max_depth,
    "min_info_gain": best_min_info_gain,
    "val_rmse": best_val_rmse,
    "cv_rmse": float(cv_avg_rmse),
    "cv_folds": NUM_FOLDS,
    "test_rmse": float(cv_test_rmse),
    "num_features": len(feature_cols),
    "tuning_strategy": f"TVS+CV{NUM_FOLDS}",
    "train_ratio": TRAIN_RATIO,
    "tuned_at": datetime.now().isoformat()
}]

spark.createDataFrame(params_data_cv).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(PARAMS_TABLE)
print(f"\n\U0001f4e6 Updated {PARAMS_TABLE} with CV score")

## ✅ Advanced Feature Engineering & Hyperparameter Tuning Complete!

### What We Built

**Feature Engineering (Cells 4–7)**:
1. **Lag Features**: price_lag_1h, 24h, 168h + load/renewable lags
2. **Rolling Statistics**: 24h mean, std, min, max + 7d load/renewable means
3. **Interaction Features**: load × renewable, load × temperature, renewable × wind
4. **Polynomial Features**: load², temperature², wind_speed²
5. **Time Features**: is_weekend, is_peak_hour, season

**Hyperparameter Tuning (Cells 8–11)**:
1. **Strategy**: TrainValidationSplit (80/20) — 3x faster than CrossValidator
2. **Grid Search**: num_trees × max_depth × min_info_gain
3. **Best Params**: Saved to `h2_tuning_best_params` Delta table
4. **Feature Importance**: Ranked by RF importance scores

### Pipeline Flow
```
05b saves params → h2_tuning_best_params (Delta table)
                        │
        ┌───────────────┴───────────────┐
        │                               │
09_MLflow_Tracking              11_Regression
(reads tuned RF params)    (reads tuned RF params)
```

### Output Tables
* `h2_gold_model_scoring_engineered` — 37 columns, consumed by 09 & 11
* `h2_tuning_best_params` — best RF hyperparameters, consumed by 09 & 11
* Prophet (12) reads `model_scoring_base` directly (univariate)